# Dish isolation - split multi-dish recordings into per-dish videos

This notebook automatically splits recordings that contain 6 dishes into separate
video files, one per dish. ROIs are selected once per session (camera is fixed)
and reused via a JSON file.

Input:  `../records/<session>/*.mp4` (or `*.mov`)
Output: `../records/<session>/isolated_dishes/<base>_dish_<i>.mp4`


## 1. Imports

**OpenCV GUI shortcuts:**
- **ENTER** or **SPACE** - confirm the selected ROI
- **c** - cancel and select again
- **q** - close the summary window (after all dishes are selected)


In [ ]:
import cv2 as cv
import sys
import json
from pathlib import Path

## 2. Path configuration

Adjust the paths here.

In [ ]:
# Session selection: 's1' (.mp4) or 's2' (.mov)
SESSION = 's1'

# Per-session config: recordings folder + video file extension
SESSIONS = {
    's1': {'dir': Path('../records/session1'), 'ext': '.mp4'},
    's2': {'dir': Path('../records/session2'), 'ext': '.mov'},
}

RECORDS_DIR = SESSIONS[SESSION]['dir']
VIDEO_EXT = SESSIONS[SESSION]['ext']
OUTPUT_DIR = RECORDS_DIR / 'isolated_dishes'
# File with saved dish ROIs (select once per session, then reuse without the GUI)
ROI_FILE = RECORDS_DIR / 'dish_rois.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Session:           {SESSION}")
print(f"Recordings folder: {RECORDS_DIR}  ({VIDEO_EXT} files)")
print(f"Output folder:     {OUTPUT_DIR}")
print(f"ROI file:          {ROI_FILE}")

## 3. Function: select the 6 dish ROIs

In [ ]:
def select_dishes_roi(video_path, num_dishes=6, max_display=1600, margin=0):
    """
    Let the user manually select an ROI for each dish.

    The preview is scaled to the MAXIMUM size that fits the screen (keeping the
    aspect ratio), not to a fixed 50%. A larger preview means a smaller error when
    mapping coordinates back to the original - this removes a systematic ROI shift.

    Args:
        video_path: path to the video file
        num_dishes: number of dishes to select (default 6)
        max_display: max preview window width/height in pixels
                     (tune to your monitor; e.g. 1600 for FullHD, 2400 for 4K)
        margin: safety margin (in original pixels) added on each side of the ROI
                so the larva is not clipped at the edge. 0 = no margin.

    Returns:
        List of (x, y, w, h) tuples in full original resolution.
    """
    capture = cv.VideoCapture(str(video_path))
    if not capture.isOpened():
        print(f'Error: cannot open video: {video_path}')
        sys.exit()

    ret, frame = capture.read()
    if not ret:
        print('Error: cannot read the first frame of the video.')
        capture.release()
        sys.exit()

    orig_h, orig_w = frame.shape[:2]

    # Fit the preview scale to the screen (keep aspect ratio, do not upscale above 1.0)
    scale = min(max_display / orig_w, max_display / orig_h, 1.0)
    display_dim = (int(orig_w * scale), int(orig_h * scale))

    print(f"\nOriginal frame: {orig_w}x{orig_h}, preview: {display_dim[0]}x{display_dim[1]} (scale {scale:.3f})")
    print(f"Select ROIs for {num_dishes} dishes.")
    print("For each dish:")
    print("  1. Drag a rectangle around the dish (with some edge margin)")
    print("  2. Press ENTER or SPACE to confirm")
    print("  3. Press 'c' to cancel and select again\n")

    rois = []
    i = 0
    while i < num_dishes:
        print(f"Select dish {i+1}/{num_dishes}...")

        display_frame = frame.copy()
        # Draw previous ROIs
        for idx, roi in enumerate(rois):
            x, y, w, h = roi
            cv.rectangle(display_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv.putText(display_frame, f'Dish {idx+1}', (x, y-10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        display_frame_resized = cv.resize(display_frame, display_dim, interpolation=cv.INTER_AREA)

        bbox = cv.selectROI(display_frame_resized)
        cv.destroyAllWindows()
        cv.waitKey(1)

        if bbox[2] > 0 and bbox[3] > 0:
            # Scale ROI coords back to the original (round instead of int = smaller error)
            x = round(bbox[0] / scale)
            y = round(bbox[1] / scale)
            w = round(bbox[2] / scale)
            h = round(bbox[3] / scale)

            # Add a safety margin and clip to frame bounds
            x0 = max(0, x - margin)
            y0 = max(0, y - margin)
            x1 = min(orig_w, x + w + margin)
            y1 = min(orig_h, y + h + margin)
            bbox_scaled = (x0, y0, x1 - x0, y1 - y0)

            rois.append(bbox_scaled)
            print(f"  [ok] Dish {i+1}: x={bbox_scaled[0]}, y={bbox_scaled[1]}, w={bbox_scaled[2]}, h={bbox_scaled[3]}")
            i += 1
        else:
            print(f"  [x] No area selected for dish {i+1}, try again...")

    # Show all selected areas
    final_frame = frame.copy()
    for idx, roi in enumerate(rois):
        x, y, w, h = roi
        cv.rectangle(final_frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
        cv.putText(final_frame, f'Dish {idx+1}', (x+10, y+30),
                   cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    final_frame_resized = cv.resize(final_frame, display_dim, interpolation=cv.INTER_AREA)
    cv.imshow('All selected dishes - press q to close', final_frame_resized)
    print("\nPress 'q' in the window to close and continue...")

    while True:
        key = cv.waitKey(1) & 0xFF
        if key == ord('q'):
            break

    cv.destroyAllWindows()
    cv.waitKey(1)
    capture.release()
    return rois

## 4. Function: extract and save individual dishes

In [ ]:
def extract_and_save_dishes(video_path, rois, output_dir, base_name):
    """
    Extract dish areas from a video and save them as separate files.

    Args:
        video_path: path to the original video file
        rois: list of (x, y, w, h) tuples for each dish
        output_dir: output folder for the saved files
        base_name: base name for the output files

    Returns:
        List of paths to the created files.
    """
    capture = cv.VideoCapture(str(video_path))
    if not capture.isOpened():
        print(f'Error: cannot open video: {video_path}')
        return []

    fps = int(capture.get(cv.CAP_PROP_FPS))
    total_frames = int(capture.get(cv.CAP_PROP_FRAME_COUNT))

    print(f"\n{'='*80}")
    print(f"Processing video: {video_path.name}")
    print(f"FPS: {fps}, frames: {total_frames}")
    print(f"Dishes to extract: {len(rois)}")
    print(f"{'='*80}\n")

    writers = []
    output_paths = []
    for i, roi in enumerate(rois):
        x, y, w, h = roi
        output_path = output_dir / f"{base_name}_dish_{i+1}.mp4"
        output_paths.append(output_path)

        fourcc = cv.VideoWriter_fourcc(*'mp4v')
        writer = cv.VideoWriter(str(output_path), fourcc, fps, (w, h))
        if not writer.isOpened():
            print(f"Error: cannot create output file: {output_path}")
            continue
        writers.append(writer)
        print(f"[ok] Prepared output file: {output_path.name}")

    print(f"\nStarting frame extraction...")
    frame_count = 0
    while True:
        ret, frame = capture.read()
        if not ret:
            break
        for roi, writer in zip(rois, writers):
            x, y, w, h = roi
            dish_frame = frame[y:y+h, x:x+w]
            writer.write(dish_frame)
        frame_count += 1
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"  Progress: {frame_count}/{total_frames} frames ({progress:.1f}%)")

    capture.release()
    for writer in writers:
        writer.release()

    print(f"\n[ok] Done! Processed {frame_count} frames.")
    print(f"[ok] Created {len(writers)} video files in: {output_dir}\n")
    return output_paths

## 5. Main function - process all recordings

In [ ]:
def load_or_select_rois(video_path, roi_file, num_dishes=6, force_reselect=False,
                        max_display=1600, margin=0):
    """Load ROIs from a JSON file; if absent, let the user select and save them.

    Args:
        video_path: recording used for ROI selection (when needed)
        roi_file: path to the JSON file with saved ROIs
        num_dishes: number of dishes
        force_reselect: force re-selection (overwrites the file)
        max_display: max preview window size (larger = more precise selection)
        margin: safety margin around the ROI in original pixels

    Returns:
        List of (x, y, w, h) tuples.
    """
    roi_file = Path(roi_file)
    if roi_file.exists() and not force_reselect:
        with open(roi_file) as f:
            rois = [tuple(r) for r in json.load(f)]
        print(f"[ok] Loaded {len(rois)} ROIs from: {roi_file.name} (no GUI)")
        return rois

    # No saved ROIs - select manually and save to JSON
    rois = select_dishes_roi(video_path, num_dishes, max_display=max_display, margin=margin)
    with open(roi_file, 'w') as f:
        json.dump(rois, f, indent=2)
    print(f"[ok] Saved {len(rois)} ROIs to: {roi_file}")
    return rois


def is_already_isolated(output_dir, base_name, num_dishes=6):
    """Check whether a recording already has a full set of isolated dishes (idempotency)."""
    return all((output_dir / f"{base_name}_dish_{i+1}.mp4").exists()
               for i in range(num_dishes))


def process_all_videos(records_dir, output_dir, roi_file, video_ext='.mp4',
                       num_dishes=6, force_reselect=False, skip_existing=True,
                       max_display=1600, margin=0):
    """
    Process all recordings in the records folder (.mp4 and .mov supported).

    - ROIs are selected/loaded ONCE (saved to JSON), same for all recordings.
    - By default, recordings that already have a full set of isolated dishes are
      skipped (so interrupted processing can be resumed without overwriting).
    """
    video_files = sorted(
        p for p in records_dir.glob('*')
        if p.suffix.lower() == video_ext.lower() and p.is_file()
    )
    if not video_files:
        print(f"No {video_ext} files found in: {records_dir}")
        return

    print(f"\n{'='*80}")
    print(f"Found {len(video_files)} video files ({video_ext}) to process:")
    print(f"{'='*80}\n")
    for i, video_file in enumerate(video_files, 1):
        status = " (already isolated - will skip)" if (skip_existing and is_already_isolated(output_dir, video_file.stem, num_dishes)) else ""
        print(f"{i}. {video_file.name}{status}")

    print(f"\n{'='*80}")
    print("Dish ROIs: load from JSON or select once.")
    print("(Assuming a fixed camera position for the whole session)")
    print(f"{'='*80}\n")
    rois = load_or_select_rois(video_files[0], roi_file, num_dishes, force_reselect,
                               max_display=max_display, margin=margin)

    all_output_files = []
    processed, skipped = 0, 0
    for i, video_path in enumerate(video_files, 1):
        base_name = video_path.stem
        if skip_existing and is_already_isolated(output_dir, base_name, num_dishes):
            print(f"\n[{i}/{len(video_files)}] Skipping (already isolated): {video_path.name}")
            skipped += 1
            continue

        print(f"\n\n{'#'*80}")
        print(f"# PROCESSING VIDEO {i}/{len(video_files)}: {video_path.name}")
        print(f"{'#'*80}")

        output_paths = extract_and_save_dishes(video_path, rois, output_dir, base_name)
        all_output_files.extend(output_paths)
        processed += 1

    print(f"\n\n{'='*80}")
    print(f"PROCESSING COMPLETE!")
    print(f"{'='*80}")
    print(f"\n[ok] Processed now: {processed} recordings")
    print(f"[skip] Skipped (existed): {skipped} recordings")
    print(f"[ok] Created: {len(all_output_files)} new dish files")
    print(f"[ok] Location: {output_dir}\n")

## 6. RUN - process all recordings

Run this cell to start processing.

On the first run, windows appear to select the 6 dish ROIs. The ROIs are **saved to
`dish_rois.json`** and loaded automatically on subsequent runs (no GUI).

**Notes:**
- Recordings that already have a **full set of 6 dishes** are **skipped** -> you can
  safely resume interrupted processing.
- To process **session II**, set `SESSION = 's2'` in the config cell and re-run
  (handles `.mov` files).
- To re-select ROIs from scratch (e.g. different camera position), set `force_reselect=True`.


In [ ]:
# Run processing for all recordings in the selected session (SESSION above).
# - max_display: preview window size. Larger = more precise selection (FullHD ~1600, 4K ~2400).
# - margin: padding (in original px) around each ROI so larvae are not clipped at the edge.
process_all_videos(
    RECORDS_DIR, OUTPUT_DIR, ROI_FILE,
    video_ext=VIDEO_EXT,
    num_dishes=6,
    force_reselect=True,
    skip_existing=True,
    max_display=1600,
    margin=20,
)

## 7. OPTIONAL - process a single recording

To process only one specific recording (e.g. for testing), use the code below.


In [ ]:
# Example: process a single file
# Adjust the file name (with the correct extension for the session: .mp4 for S1, .mov for S2)
video_name = f"coli_5x10_8{VIDEO_EXT}"
video_path = RECORDS_DIR / video_name
base_name = video_path.stem

# ROIs: load from the saved file (or select if absent)
rois = load_or_select_rois(video_path, ROI_FILE, num_dishes=6, max_display=1600, margin=20)

output_paths = extract_and_save_dishes(video_path, rois, OUTPUT_DIR, base_name)

print(f"\n[ok] Created {len(output_paths)} files:")
for path in output_paths:
    print(f"  - {path.name}")

## 8. Verify the results

Check how many files were created and their sizes:

In [ ]:
from collections import defaultdict

output_files = sorted(list(OUTPUT_DIR.glob('*.mp4')))
print(f"[ok] Created {len(output_files)} video files\n")

# Group files by source recording
grouped_files = defaultdict(list)
for file in output_files:
    # Extract the base name (without "_dish_X.mp4")
    base = '_'.join(file.stem.split('_')[:-2])
    grouped_files[base].append(file)

print("Files grouped by source recording:\n")
for base_name, files in sorted(grouped_files.items()):
    print(f"{base_name}:")
    for file in sorted(files):
        file_size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - {file.name} ({file_size_mb:.2f} MB)")
    print()